# Scikit Learn Pipelines 
- Chaining multiple steps so that the output of each step is used as input to the next step.
- Makes it easy to apply the same preprocessing to train and test.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
df= pd.read_csv('../data/tested.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,34.5,0,0,7.8292,Q
1,1,3,female,47.0,1,0,7.0000,S
2,0,2,male,62.0,0,0,9.6875,Q
3,0,3,male,27.0,0,0,8.6625,S
4,1,3,female,22.0,1,1,12.2875,S


In [7]:
X_train,X_test,y_train,y_test= train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size=0.2,
    random_state=42
)

In [8]:
X_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0,S
31,2,male,24.0,2,0,31.5,S


In [9]:
y_train.head(3)

336    0
31     0
84     0
Name: Survived, dtype: int64

In [10]:
df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

In [22]:
#Applying Imputation

si_age=SimpleImputer()
X_train_age=si_age.fit_transform(X_train[['Age']])
X_test_age= si_age.transform(X_test[['Age']])

In [23]:
X_train_age.shape

(334, 1)

In [24]:
#One Hot Encoding for sex and Embarked 
ohe_sex_embarked= OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_sex_embarked=ohe_sex_embarked.fit_transform(X_train[['Sex','Embarked']])
X_test_sex_embarked=ohe_sex_embarked.fit_transform(X_test[['Sex','Embarked']])



In [25]:
X_train_sex_embarked.shape

(334, 5)

In [29]:
X_train_rem=X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem=X_test.drop(columns=['Sex','Age','Embarked'])


In [32]:
X_train_transformed= np.concatenate((X_train_rem,X_train_age,X_train_sex_embarked),axis=1)
X_test_transformed= np.concatenate((X_test_rem,X_test_age,X_test_sex_embarked),axis=1)

In [35]:
X_test_transformed.shape

(84, 10)

In [36]:
X_train_transformed.shape

(334, 10)

In [37]:
#Decision tree classifier
clf= DecisionTreeClassifier()
clf.fit(X_train_transformed,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [38]:
y_pred= clf.predict(X_test_transformed)

In [40]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

1.0

In [41]:
import pickle

In [44]:
pickle.dump(ohe_sex_embarked,open('../data/ohe_sex_embarked.pkl','wb'))
pickle.dump(clf,open('../data/clf.pkl','wb'))